# Synthetic Data Generation

This notebook demonstrates both modes of synthetic data generation:
1. **Ground Truth Mode**: Generate from existing studies with PPTX + Excel files
2. **No Ground Truth Mode**: Test new concepts without historical data

## Setup

In [ ]:
import sys
import os
import json
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# Setup paths
project_root = Path.cwd().parent
sys.path.append(str(project_root))
load_dotenv(project_root / '.env')

from src.kantar.survey_runner import KantarSurveyRunner
from src.kantar.study_catalog import StudyCatalog
from src.kantar.market_profiles import list_market_profiles

print("✓ Environment loaded")
print(f"✓ Project root: {project_root}")

## Mode 1: Ground Truth Generation

Generate synthetic data for an existing study. The system will:
1. Load concepts from PPTX files
2. Extract ground truth Excel structure
3. Generate personas (optionally from GT demographics)
4. Create responses matching GT format

### Select a Study and Market

In [ ]:
# List available studies
catalog = StudyCatalog()

print("Available Studies:")
for i, study in enumerate(catalog.studies.values(), 1):
    print(f"{i}. {study.study_id}: {study.study_name}")
    print(f"   Markets: {', '.join(study.complete_markets)}\n")

# Select first study with complete markets
study = None
for s in catalog.studies.values():
    if s.complete_markets:
        study = s
        break

if not study:
    print("⚠️  No complete studies found. Please add study data to data/kantar-survey-source/")
else:
    market = study.complete_markets[0]
    print(f"Selected: {study.study_id} - {market}")

### Configuration

In [ ]:
# Generation parameters
if study:
    config = {
        'study_id': study.study_id,
        'market_code': market,
        'num_respondents': 20,  # Start small, increase to 100-200 for full study
        'model': 'gpt-4o-mini',  # Fast and cost-effective
        'use_ground_truth_demographics': True  # Sample demographics from GT
    }
    
    print("Generation Configuration:")
    for key, value in config.items():
        print(f"  {key}: {value}")
else:
    print("⚠️  Skipping - no study selected")

### Generate Synthetic Data

In [ ]:
# Initialize runner
if study:
    runner = KantarSurveyRunner(model=config['model'])
    
    print(f"Generating {config['num_respondents']} respondents...")
    print("This may take 5-10 minutes depending on respondent count...\n")
    
    # Generate
    output_file = runner.generate_for_market(
        study_id=config['study_id'],
        market_code=config['market_code'],
        num_respondents=config['num_respondents'],
        use_ground_truth_demographics=config['use_ground_truth_demographics']
    )
    
    print(f"\n✓ Generation complete!")
    print(f"  Output: {output_file}")
else:
    print("⚠️  Skipping generation - no study available")

### Preview Generated Data

In [ ]:
# Load generated data
if study and 'output_file' in locals():
    df = pd.read_excel(output_file)
    
    print(f"Dataset: {len(df)} rows × {len(df.columns)} columns\n")
    
    # Demographics
    print("Demographics Distribution:")
    print("=" * 60)
    if 'Gender' in df.columns:
        print("\nGender:")
        print(df['Gender'].value_counts())
    
    if 'AGE' in df.columns:
        print("\nAge Statistics:")
        print(f"  Min: {df['AGE'].min()}")
        print(f"  Max: {df['AGE'].max()}")
        print(f"  Mean: {df['AGE'].mean():.1f}")
        print(f"  Median: {df['AGE'].median():.0f}")
    
    # Question responses
    print("\nSample Question Responses:")
    print("=" * 60)
    question_cols = [c for c in df.columns if '(' in c and ')' in c][:3]
    for col in question_cols:
        print(f"\n{col}:")
        print(df[col].value_counts().head(5))
else:
    print("⚠️  No data to preview - generate data first")

## Mode 2: No Ground Truth Generation

Test brand new concepts without PPTX files or historical data. Perfect for:
- Early-stage concept testing
- What-if scenarios
- Quick prototyping

Uses pre-built market profiles for demographics.

### View Available Market Profiles

In [ ]:
print("Available Market Profiles:")
print("=" * 60)
for profile_id, info in list_market_profiles().items():
    print(f"\n{profile_id}:")
    print(f"  {info['name']}")
    print(f"  {info['description']}")

### Define New Concepts

In [ ]:
# Define your concepts
new_concepts = [
    {
        'id': 'TestConcept1',
        'name': 'Premium Lottery Experience',
        'description': 'A new premium lottery game with enhanced odds and exclusive VIP prizes',
        'price': '$10 per ticket',
        'features': [
            'Enhanced winning odds (1 in 100 vs standard 1 in 200)',
            'Exclusive VIP prize tiers',
            'Monthly member benefits',
            'Priority customer support'
        ],
        'occasion': 'Regular play for serious players'
    },
    {
        'id': 'TestConcept2',
        'name': 'Quick Pick Plus',
        'description': 'Instant lottery with AI-powered number selection and immediate results',
        'price': '$2 per ticket',
        'features': [
            'AI-powered smart number selection',
            'Instant results within 60 seconds',
            'Mobile-first experience',
            'Lower price point for casual play'
        ],
        'occasion': 'Quick entertainment'
    },
    {
        'id': 'TestConcept3',
        'name': 'Social Jackpot',
        'description': 'Group lottery experience where friends pool tickets and share winnings',
        'price': '$5 per group share',
        'features': [
            'Create or join lottery pools with friends',
            'Automated prize distribution',
            'Social leaderboards',
            'Group chat and notifications'
        ],
        'occasion': 'Social entertainment'
    }
]

print(f"Defined {len(new_concepts)} new concepts:")
for concept in new_concepts:
    print(f"  - {concept['name']} ({concept['price']})")

### Generate from New Concepts

In [ ]:
# Configuration
no_gt_config = {
    'concepts': new_concepts,
    'num_respondents': 50,
    'market_profile': 'US_gaming',  # or 'UK_lottery', 'EU_general', 'generic'
    'model': 'gpt-4o-mini',
    'output_name': 'test_concepts_initial'
}

print("No-Ground-Truth Configuration:")
print(f"  Concepts: {len(no_gt_config['concepts'])}")
print(f"  Respondents: {no_gt_config['num_respondents']}")
print(f"  Market Profile: {no_gt_config['market_profile']}")
print(f"  Model: {no_gt_config['model']}")

# Initialize runner
runner_no_gt = KantarSurveyRunner(model=no_gt_config['model'])

print(f"\nGenerating synthetic data for new concepts...")
print("This may take 10-15 minutes...\n")

# Generate
output_file_no_gt = runner_no_gt.generate_from_concepts(
    concepts=no_gt_config['concepts'],
    num_respondents=no_gt_config['num_respondents'],
    market_profile=no_gt_config['market_profile'],
    output_name=no_gt_config['output_name']
)

print(f"\n✓ Generation complete!")
print(f"  Output: {output_file_no_gt}")

### Analyze Generated Data

In [ ]:
# Load data
df_no_gt = pd.read_excel(output_file_no_gt)

print(f"Dataset: {len(df_no_gt)} rows × {len(df_no_gt.columns)} columns\n")

# Analyze responses by concept
print("Purchase Intent by Concept:")
print("=" * 60)

pi_cols = [c for c in df_no_gt.columns if 'PRPURINT' in c or 'PURINT' in c]
if pi_cols:
    for col in pi_cols:
        concept_name = col.split('-')[-1].strip() if '-' in col else 'Unknown'
        print(f"\n{concept_name}:")
        print(df_no_gt[col].value_counts().sort_index())

# Uniqueness scores
print("\n\nUniqueness by Concept:")
print("=" * 60)

uniq_cols = [c for c in df_no_gt.columns if 'UNIQ' in c]
if uniq_cols:
    for col in uniq_cols:
        concept_name = col.split('-')[-1].strip() if '-' in col else 'Unknown'
        mean_score = df_no_gt[col].dropna().apply(lambda x: int(x.split('(')[1].split(')')[0]) if '(' in str(x) else None).mean()
        print(f"  {concept_name}: {mean_score:.2f}")

## Saving Concepts for Later Use

Save your concepts to JSON for reuse:

In [ ]:
# Save concepts to file
concepts_file = project_root / 'my_concepts.json'

with open(concepts_file, 'w') as f:
    json.dump(new_concepts, f, indent=2)

print(f"✓ Saved concepts to: {concepts_file}")
print("\nReuse with CLI:")
print(f"python -m src.kantar.survey_runner \\")
print(f"  --concepts-file {concepts_file.name} \\")
print(f"  --num-respondents 100 \\")
print(f"  --market-profile US_gaming \\")
print(f"  --model gpt-4o-mini")

## Summary

You now know how to:
- ✅ Generate from existing studies (ground truth mode)
- ✅ Generate from custom concepts (no-ground-truth mode)
- ✅ Configure demographics (GT sampling vs market profiles)
- ✅ Analyze generated data
- ✅ Save concepts for reuse

**Next Steps:**
- Notebook 03: Validate against ground truth and generate reports
- Scale up respondent counts for production datasets
- Compare different concept variations